<hr style=\"color:#5A7D9F;\">
<p align=\"center\">
    <b style=\"font-size:2vw; color:#5A7D9F; font-weight:bold;\">
    <center>Benthic Traits Mapping</center>
    </b>
</p>
<hr style=\"color:#5A7D9F;\">

A simple notebook to analyze our data, do plots and save them.

In [ ]:
# ---------- Librairies ----------
import math
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from benthic.config import PATH_DATASET_TRAIN_VAL
from benthic.data.const import DATASET_INPUT_VARIABLES, DATASET_OUTPUT_VARIABLES
from benthic.data.tools import Statistics, from_zarr_to_numpy
from pathlib import Path
from scipy.stats import gaussian_kde

# ---------- Jupyter ----------
%matplotlib inline
plt.rcParams.update({"font.size": 13})

# Making sure modules are reloaded when modified
%reload_ext autoreload
%autoreload 2

In [ ]:
# Creating repositories to save plot
PLOTS_DIR = Path("plots")
PLOTS_DIR.mkdir(exist_ok=True)
MARGINALS_DIR = PLOTS_DIR / "marginals"
JOINTS_DIR = PLOTS_DIR / "joints"
VARIABILITY_DIR = PLOTS_DIR / "variability"
VARIABILITY_DIR.mkdir(parents=True, exist_ok=True)
for d in [MARGINALS_DIR, JOINTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Loading the dataset
ds = xr.open_zarr(PATH_DATASET_TRAIN_VAL)

# Extracting the input and output variables
X, y = (
    from_zarr_to_numpy(ds, DATASET_INPUT_VARIABLES),
    from_zarr_to_numpy(ds, DATASET_OUTPUT_VARIABLES),
)
print(f"Input shape: {X.shape}", f" | Output shape: {y.shape}")

# Standardizing data
stats_X, stats_y = Statistics(), Statistics()

stats_X.fit(X)
stats_y.fit(y)

X_standardized = stats_X.standardize(X)
y_standardized = stats_y.standardize(y)

# Expanded input labels (temporal vars → var_mean/std/min/max, static vars → var)
STATS = ["mean", "std", "min", "max"]
X_LABELS = []
for var in DATASET_INPUT_VARIABLES:
    if "time" in ds[var].dims:
        X_LABELS.extend([f"{var}_{s}" for s in STATS])
    else:
        X_LABELS.append(var)

print(f"X labels ({len(X_LABELS)}): {X_LABELS}")

<hr style="color:#5A7D9F;">
<p align="center">
    <b style="font-size:2vw; color:#5A7D9F; font-weight:bold;">
    <center>Marginals & Joint Distributions</center>
    </b>
</p>
<hr style="color:#5A7D9F;">

In [ ]:
def plot_marginal(data, label, save_path):
    r"""Save a 1D marginal plot: histogram + KDE curve."""
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.hist(data, bins=25, density=True, color="steelblue", alpha=0.4)
    try:
        kde = gaussian_kde(data)
        xg = np.linspace(data.min(), data.max(), 300)
        ax.plot(xg, kde(xg), color="steelblue", linewidth=2)
    except Exception:
        pass
    ax.set_xlabel(label, fontsize=12)
    ax.set_ylabel("Density", fontsize=12)
    ax.set_title(f"p({label})", fontsize=13)
    ax.tick_params(labelsize=10)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)


# Input marginals
for j, label in enumerate(X_LABELS):
    plot_marginal(X_standardized[:, j], label, MARGINALS_DIR / f"x_{label}.png")

# Output marginals
for i, label in enumerate(DATASET_OUTPUT_VARIABLES):
    plot_marginal(y_standardized[:, i], label, MARGINALS_DIR / f"y_{label}.png")

print(f"Saved {len(X_LABELS) + len(DATASET_OUTPUT_VARIABLES)} marginals → {MARGINALS_DIR}")

In [ ]:
# Preparing for joint plots
n_y = y_standardized.shape[1]
n_cols = math.ceil(math.sqrt(n_y))
n_rows = math.ceil(n_y / n_cols)


def plot_joints(x_data, x_label, y_data, y_labels, save_path):
    r"""Save a grid of joint density plots p(y_i, x) for all output variables."""
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.5 * n_cols, 2.5 * n_rows))
    axes = axes.ravel()

    for i, (yi, label) in enumerate(zip(y_data.T, y_labels)):
        ax = axes[i]
        try:
            kde = gaussian_kde(np.vstack([x_data, yi]))
            xg, yg = (
                np.linspace(x_data.min(), x_data.max(), 60),
                np.linspace(yi.min(), yi.max(), 60),
            )
            Xg, Yg = np.meshgrid(xg, yg)
            Z = kde(np.vstack([Xg.ravel(), Yg.ravel()])).reshape(Xg.shape)
            ax.contourf(Xg, Yg, Z, levels=8, cmap="Blues")
            ax.contour(Xg, Yg, Z, levels=8, colors="steelblue", linewidths=0.5, alpha=0.8)
        except Exception:
            ax.scatter(x_data, yi, alpha=0.3, s=2, color="steelblue")
        ax.set_xlabel(x_label, fontsize=7)
        ax.set_ylabel(label, fontsize=7)
        ax.tick_params(labelsize=6)

    # Hide unused subplots
    for i in range(n_y, len(axes)):
        axes[i].set_visible(False)

    fig.suptitle(f"p(y, {x_label})", fontsize=13)
    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved → {save_path}")


for j, x_label in enumerate(X_LABELS):
    plot_joints(
        x_data=X_standardized[:, j],
        x_label=x_label,
        y_data=y_standardized,
        y_labels=DATASET_OUTPUT_VARIABLES,
        save_path=JOINTS_DIR / f"{x_label}.png",
    )

<hr style="color:#5A7D9F;">
<p align="center">
    <b style="font-size:2vw; color:#5A7D9F; font-weight:bold;">
    <center>Spatial Variability Analysis</center>
    </b>
</p>
<hr style="color:#5A7D9F;">

In [ ]:
# Day-of-year indices (0-indexed) — meteorological seasons
SEASONS_DAYS = {
    "Winter": list(range(0, 59)) + list(range(334, 365)),  # Jan–Feb + Dec
    "Spring": list(range(59, 151)),  # Mar–May
    "Summer": list(range(151, 243)),  # Jun–Aug
    "Autumn": list(range(243, 334)),  # Sep–Nov
}

MONTHS_DAYS = {
    "Jan": range(0, 31),
    "Feb": range(31, 59),
    "Mar": range(59, 90),
    "Apr": range(90, 120),
    "May": range(120, 151),
    "Jun": range(151, 181),
    "Jul": range(181, 212),
    "Aug": range(212, 243),
    "Sep": range(243, 273),
    "Oct": range(273, 304),
    "Nov": range(304, 334),
    "Dec": range(334, 365),
}

SEASON_COLORS = {
    "Winter": "steelblue",
    "Spring": "mediumseagreen",
    "Summer": "tomato",
    "Autumn": "darkorange",
}
MONTH_COLORS = [cm.hsv(i / 12) for i in range(12)]


def plot_kde(ax, data, color, label, linewidth=1.5, alpha=1.0):
    r"""Plot a KDE curve on ax (silently skips if singular covariance)."""
    try:
        kde = gaussian_kde(data)
        xg = np.linspace(data.min(), data.max(), 300)
        ax.plot(xg, kde(xg), color=color, linewidth=linewidth, alpha=alpha, label=label)
    except Exception:
        pass


def plot_temporal(ax, var, variable):
    r"""Plot temporal evolution of all stations on ax."""
    for station in range(var.shape[0]):
        ax.plot(var[station, :], alpha=0.5, linewidth=0.8)
    ax.set_xlabel("Day of year", fontsize=14)
    ax.set_ylabel(variable, fontsize=14)
    ax.set_title("Temporal evolution per station", fontsize=15, fontweight="bold")
    ax.tick_params(labelsize=12)


def save_fig(fig, path):
    r"""Save figure as PDF and close it."""
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved → {path}")


for variable in DATASET_INPUT_VARIABLES[3:]:
    var = ds[variable].values  # (n_stations, 365)

    # ---- Figure 1: Annual mean ----
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    plot_temporal(axes[0], var, variable)
    ax = axes[1]
    ax.hist(var.mean(axis=1), bins=20, density=True, color="gray", alpha=0.25)
    plot_kde(ax, var.mean(axis=1), color="black", label="Annual", linewidth=2.5)
    ax.set_xlabel(variable, fontsize=14)
    ax.set_ylabel("Density", fontsize=14)
    ax.set_title("Annual mean distribution", fontsize=15, fontweight="bold")
    ax.legend(fontsize=12)
    ax.tick_params(labelsize=12)
    fig.suptitle(f"{variable.capitalize()} — Annual", fontsize=16, fontweight="bold")
    plt.tight_layout()
    save_fig(fig, VARIABILITY_DIR / f"{variable}_annual.pdf")

    # ---- Figure 2: Seasonal means ----
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    plot_temporal(axes[0], var, variable)
    ax = axes[1]
    for season, days in SEASONS_DAYS.items():
        plot_kde(
            ax, var[:, days].mean(axis=1), color=SEASON_COLORS[season], label=season, linewidth=2.0
        )
    ax.set_xlabel(variable, fontsize=14)
    ax.set_ylabel("Density", fontsize=14)
    ax.set_title("Seasonal mean distributions", fontsize=15, fontweight="bold")
    ax.legend(fontsize=12)
    ax.tick_params(labelsize=12)
    fig.suptitle(f"{variable.capitalize()} — Seasonal", fontsize=16, fontweight="bold")
    plt.tight_layout()
    save_fig(fig, VARIABILITY_DIR / f"{variable}_seasonal.pdf")

    # ---- Figure 3: Monthly means ----
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    plot_temporal(axes[0], var, variable)
    ax = axes[1]
    for (month, days), color in zip(MONTHS_DAYS.items(), MONTH_COLORS):
        plot_kde(
            ax, var[:, days].mean(axis=1), color=color, label=month, linewidth=1.5, alpha=0.85
        )
    ax.set_xlabel(variable, fontsize=14)
    ax.set_ylabel("Density", fontsize=14)
    ax.set_title("Monthly mean distributions", fontsize=15, fontweight="bold")
    ax.legend(fontsize=10, ncol=2)
    ax.tick_params(labelsize=12)
    fig.suptitle(f"{variable.capitalize()} — Monthly", fontsize=16, fontweight="bold")
    plt.tight_layout()
    save_fig(fig, VARIABILITY_DIR / f"{variable}_monthly.pdf")